# Tapioca Gap Analysis

For each store and date, measures the **intra-day time gaps** between consecutive tapioca sales.

A long gap mid-day suggests the employee ran out or didn't prepare more tapioca.

Each gap row shows:
- **gap_start / gap_end** — timestamps of the tapioca sales that bookend the gap
- **gap_minutes** — how long without a tapioca sale
- **revenue_during_gap** — total store revenue sold during that window (proxy for missed tapioca attach)

In [ ]:
# ── Config ────────────────────────────────────────────────────────

from pipeline.config import TRANSACTIONS_TABLE

# Filter keyword for tapioca products (case-insensitive)
TAPIOCA_KEYWORD = "тапиок"  # matches тапиока, тапиоки etc.

# Date range to analyse (None = all available data)
DATE_FROM = None  # e.g. "2025-01-01"
DATE_TO   = None  # e.g. "2025-12-31"

# Output CSV path (Databricks-accessible)
OUTPUT_CSV = "/tmp/tapioca_gap_analysis.csv"

In [ ]:
# ── Load Transactions ─────────────────────────────────────────────

from pipeline.transforms import _get_spark
from pyspark.sql import functions as F
import pandas as pd

spark = _get_spark()

sdf = spark.table(TRANSACTIONS_TABLE)
if DATE_FROM:
    sdf = sdf.filter(F.col("date") >= DATE_FROM)
if DATE_TO:
    sdf = sdf.filter(F.col("date") <= DATE_TO)

df = sdf.toPandas()
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df):,} rows | {df['date'].min().date()} → {df['date'].max().date()}")

In [ ]:
# ── Identify Tapioca Transactions ────────────────────────────────

tapioca_mask = (
    df["product"].str.contains(TAPIOCA_KEYWORD, case=False, na=False)
    & (df["qty"] > 0)
    & (~df["is_return"])
)

tapioca_txns = (
    df[tapioca_mask][["store_name", "date", "datetime", "qty"]]
    .sort_values(["store_name", "date", "datetime"])
    .reset_index(drop=True)
)

print(f"Tapioca transactions: {len(tapioca_txns):,}")

# Preview matching product names to confirm keyword
matched_products = df[tapioca_mask]["product"].unique()
print(f"\nMatched products ({len(matched_products)}):")
for p in sorted(matched_products):
    print(f"  {p}")

In [ ]:
# ── Compute Intra-Day Gaps Between Tapioca Sales ─────────────────

MIN_GAP_MINUTES = 60  # only report gaps of 1 hour or more

gaps = []

for (store, date), group in tapioca_txns.groupby(["store_name", "date"]):
    group = group.sort_values("datetime").reset_index(drop=True)

    for i in range(1, len(group)):
        t_start = group.loc[i - 1, "datetime"]
        t_end   = group.loc[i,     "datetime"]
        gap_min = (t_end - t_start).total_seconds() / 60

        if gap_min < MIN_GAP_MINUTES:
            continue

        # Revenue from all sales strictly within the gap window
        between = df[
            (df["store_name"] == store)
            & (df["datetime"] > t_start)
            & (df["datetime"] < t_end)
            & (~df["is_return"])
        ]
        revenue = round(between["revenue"].sum())

        gaps.append({
            "store":              store,
            "date":               date,
            "gap_start":          t_start,
            "gap_end":            t_end,
            "gap_minutes":        round(gap_min),
            "revenue_during_gap": revenue,
        })

gaps_df = (
    pd.DataFrame(gaps)
    .sort_values(["store", "date", "gap_minutes"], ascending=[True, True, False])
    .reset_index(drop=True)
)

print(f"Found {len(gaps_df):,} gaps ≥ {MIN_GAP_MINUTES} min across {gaps_df['store'].nunique()} stores")
print(f"Avg gap: {gaps_df['gap_minutes'].mean():.0f} min | Max gap: {gaps_df['gap_minutes'].max()} min")
gaps_df.head(20)

In [ ]:
# ── Summary Stats ─────────────────────────────────────────────────

summary = (
    gaps_df.groupby("store").agg(
        total_gaps         = ("gap_minutes",        "count"),
        avg_gap_minutes    = ("gap_minutes",        "mean"),
        max_gap_minutes    = ("gap_minutes",        "max"),
        total_revenue_lost = ("revenue_during_gap", "sum"),
        avg_revenue_lost   = ("revenue_during_gap", "mean"),
    )
    .round(1)
    .sort_values("total_revenue_lost", ascending=False)
)

print("Gap summary by store:")
summary

In [ ]:
# ── Save CSV ──────────────────────────────────────────────────────

gaps_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(gaps_df):,} rows to {OUTPUT_CSV}")

# Also save to repo data folder if running locally
import os
local_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data', 'tapioca_gap_analysis.csv')
os.makedirs(os.path.dirname(local_path), exist_ok=True)
gaps_df.to_csv(local_path, index=False)
print(f"Also saved to {local_path}")

In [ ]:
# ── Visualization ─────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

stores = sorted(gaps_df["store"].unique())
n = len(stores)
cols = 2
rows = (n + 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
axes = axes.flatten()

for i, store in enumerate(stores):
    ax = axes[i]
    sg = gaps_df[gaps_df["store"] == store].copy()
    sg["date"] = pd.to_datetime(sg["date"])

    # Scatter: x=date, y=gap_minutes, size=revenue during gap
    sc = ax.scatter(
        sg["date"],
        sg["gap_minutes"],
        c=sg["revenue_during_gap"],
        s=sg["revenue_during_gap"].clip(lower=100) / 20,
        cmap="RdYlGn_r",
        alpha=0.7,
        edgecolors="white",
        linewidths=0.4,
    )

    # Reference line: 60 min gap threshold
    ax.axhline(60, color="red", linestyle="--", linewidth=0.8, alpha=0.6, label="60 min threshold")

    ax.set_title(store, fontsize=11, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Gap length (minutes)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
    ax.legend(fontsize=7)
    fig.colorbar(sc, ax=ax, label="Revenue in gap (₽)")

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Intra-Day Tapioca Gaps by Store\n(bubble size = revenue earned while tapioca was unavailable)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/tapioca_gap_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to /tmp/tapioca_gap_analysis.png")